# Data compression

In [1]:
import duckdb
import os
import time
import pandas as pd

con = duckdb.connect()
con.execute("SET memory_limit='3GB';")
con.execute("SET threads=2;")
con.execute("SET temp_directory='/work/data/tmp';")

### Data Storage Strategy: CSV → Parquet Conversion

Before performing any analysis, we convert all raw CSV files to Parquet format.

**Why this step is necessary:**

- The largest fact table (not even considering web sessions) contains ~12.5 million rows and several high-cardinality string columns.
- Loading CSV files directly into pandas risks exceeding Deepnote’s 5GB RAM limit.
- CSV is a row-based text format that must be parsed every time it is read.
- Parquet is a compressed, columnar format that:
  - Is significantly smaller on disk
  - Loads much faster
  - Preserves data types
  - Allows column pruning (only required columns are read)

After conversion, the CSV files are treated as raw archival data and are no longer used for analysis.

**Workflow design:**

CSV (raw data) → Parquet (analysis layer) → DuckDB (aggregation & joins) → pandas (modeling & final analysis)

This approach ensures memory stability, faster queries, and reproducible processing within Deepnote’s resource constraints.


### Correct datatypes

In [2]:
schemas = {
    "fact_sales_v2.csv": """
        matas_id::VARCHAR AS matas_id,
        transaction_id::VARCHAR AS transaction_id,
        store_id::VARCHAR AS store_id,
        channel::VARCHAR AS channel,
        order_id::VARCHAR AS order_id,
        order_delivery_method::VARCHAR AS order_delivery_method,
        sales_date::DATE AS sales_date,
        product_id::VARCHAR AS product_id,
        product_quantity::INTEGER AS product_quantity,
        revenue::DOUBLE AS revenue,
        discount::DOUBLE AS discount
    """,

    "dim_store_v2.csv": """
        store_id::VARCHAR AS store_id,
        store_name::VARCHAR AS store_name,
        store_zip_code::VARCHAR AS store_zip_code,
        store_region::VARCHAR AS store_region,
        is_webshop::BOOLEAN AS is_webshop
    """,

    "dim_product_v2.csv": """
        product_id::VARCHAR AS product_id,
        product_name::VARCHAR AS product_name,
        brand_name::VARCHAR AS brand_name,
        category::VARCHAR AS category,
        sub_category::VARCHAR AS sub_category,
        segment::VARCHAR AS segment,
        sub_segment::VARCHAR AS sub_segment,
        contents_quantity::DOUBLE AS contents_quantity,
        contents_unit::VARCHAR AS contents_unit,
        sales_price::DOUBLE AS sales_price,
        attributes::VARCHAR AS attributes
    """,

    "dim_campaign_v2.csv": """
        campaign_id::VARCHAR AS campaign_id,
        campaign_name::VARCHAR AS campaign_name,
        campaign_type::VARCHAR AS campaign_type,
        campaign_start_date::DATE AS campaign_start_date,
        campaign_end_date::DATE AS campaign_end_date,
        campaign_primary_categories::VARCHAR AS campaign_primary_categories,
        discounts::VARCHAR AS discounts
    """,

    "dim_member_v2.csv": """
        matas_id::VARCHAR AS matas_id,
        registration_date::DATE AS registration_date,
        gender::VARCHAR AS gender,
        age::INTEGER AS age,
        age_group::VARCHAR AS age_group,
        region::VARCHAR AS region,
        customer_type::VARCHAR AS customer_type,
        marketing_segment::VARCHAR AS marketing_segment,
        value_segment::VARCHAR AS value_segment,
        matas_plus::BOOLEAN AS matas_plus,
        my_brands::VARCHAR AS my_brands,
        app_user::BOOLEAN AS app_user,
        emails_received::INTEGER AS emails_received,
        emails_opened::INTEGER AS emails_opened,
        emails_clicked::INTEGER AS emails_clicked,
        last_email_open::DATE AS last_email_open,
        points::INTEGER AS points,
        latest_points_earned_date::DATE AS latest_points_earned_date,
        yearly_spend::VARCHAR AS yearly_spend,
        frequency::VARCHAR AS frequency,
        campaign_sales_kr::DOUBLE AS campaign_sales_kr,
        total_sales_kr::DOUBLE AS total_sales_kr,
        total_transactions::INTEGER AS total_transactions,
        total_basket_size::DOUBLE AS total_basket_size,
        total_last_purchase::DATE AS total_last_purchase,
        store_sales_kr::DOUBLE AS store_sales_kr,
        store_transactions::INTEGER AS store_transactions,
        store_basket_size::DOUBLE AS store_basket_size,
        store_last_purchase::DATE AS store_last_purchase,
        web_sales_kr::DOUBLE AS web_sales_kr,
        web_transactions::INTEGER AS web_transactions,
        web_basket_size::DOUBLE AS web_basket_size,
        web_last_purchase::DATE AS web_last_purchase
    """,

    "fact_campaign_sales_v2.csv": """
        campaign_id::VARCHAR AS campaign_id,
        matas_id::VARCHAR AS matas_id,
        sales_date::DATE AS sales_date,
        store_id::VARCHAR AS store_id,
        channel::VARCHAR AS channel,
        transaction_id::VARCHAR AS transaction_id,
        product_id::VARCHAR AS product_id,
        revenue::DOUBLE AS revenue,
        product_quantity::INTEGER AS product_quantity
    """,

    "fact_points_v2.csv": """
        matas_id::VARCHAR AS matas_id,
        point_date::DATE AS point_date,
        point_type::VARCHAR AS point_type,
        points::INTEGER AS points
    """,

    "fact_web_sessions_v2.csv": """
        matas_id::VARCHAR AS matas_id,
        session_id::VARCHAR AS session_id,
        platform::VARCHAR AS platform,
        event_date::DATE AS event_date,
        event_timestamp::TIMESTAMP AS event_timestamp,
        event_name::VARCHAR AS event_name,
        item_ids::VARCHAR AS item_ids,
        page_view_brand::VARCHAR AS page_view_brand,
        page_view_category::VARCHAR AS page_view_category
    """
}

### Writing parquets

In [3]:
csv_files = [
    "/work/data/dim_campaign_v2.csv",
    "/work/data/dim_member_v2.csv",
    "/work/data/dim_product_v2.csv",
    "/work/data/dim_store_v2.csv",
    "/work/data/fact_campaign_sales_v2.csv",
    "/work/data/fact_points_v2.csv",
    "/work/data/fact_sales_v2.csv",
    "/datasets/factwebsessionsv2/fact_web_sessions_v2.csv"
]


output_dir = "/work/data/parquets"

# Ensure folder exists
os.makedirs(output_dir, exist_ok=True)

print("Starting conversion to Parquet using DuckDB...\n")

for file in csv_files:

    filename = os.path.basename(file)
    output_path = os.path.join(output_dir, filename.replace(".csv", ".parquet"))

    schema_sql = schemas.get(filename)

    if not schema_sql:
        raise ValueError(f"No schema defined for {filename}")

    print(f"Processing {filename}...")

    start = time.perf_counter()

    con.execute(f"""
    COPY (
        SELECT {schema_sql}
        FROM read_csv_auto('{file}')
    )
    TO '{output_path}'
    (FORMAT 'parquet', COMPRESSION 'zstd');
    """)

    end = time.perf_counter()

    print(f"Finished {filename} in {end - start:.2f}s")

Starting conversion to Parquet using DuckDB...

Processing dim_campaign_v2.csv...
Finished dim_campaign_v2.csv in 8.87s
Processing dim_member_v2.csv...
Finished dim_member_v2.csv in 15.01s
Processing dim_product_v2.csv...
Finished dim_product_v2.csv in 6.88s
Processing dim_store_v2.csv...
Finished dim_store_v2.csv in 4.71s
Processing fact_campaign_sales_v2.csv...
Finished fact_campaign_sales_v2.csv in 27.48s
Processing fact_points_v2.csv...
Finished fact_points_v2.csv in 28.15s
Processing fact_sales_v2.csv...
Finished fact_sales_v2.csv in 98.29s
Processing fact_web_sessions_v2.csv...
Finished fact_web_sessions_v2.csv in 145.84s


Testing resulting file sizes\. If the largest file is ~500MB, that should be completely manageable in 5GB RAM\.

In [4]:
parquet_dir = "/work/data/parquets"

for f in os.listdir(parquet_dir):
    size_mb = os.path.getsize(os.path.join(parquet_dir, f)) / 1e6
    print(f"{f}: {size_mb:.2f} MB")

fact_campaign_sales_v2.parquet: 67.96 MB
dim_store_v2.parquet: 0.01 MB
fact_sales_v2.parquet: 337.33 MB
fact_web_sessions_v2.parquet: 480.36 MB
dim_campaign_v2.parquet: 7.98 MB
dim_member_v2.parquet: 35.16 MB
fact_points_v2.parquet: 35.25 MB
dim_product_v2.parquet: 1.83 MB


### Checking final data types

In [20]:
PATH_PARQUETS = "/work/data/parquets"

fact_sales = pl.scan_parquet(os.path.join(PATH_PARQUETS, "fact_sales_v2.parquet"))
fact_campaign_sales = pl.scan_parquet(os.path.join(PATH_PARQUETS, "fact_campaign_sales_v2.parquet"))
fact_points = pl.scan_parquet(os.path.join(PATH_PARQUETS, "fact_points_v2.parquet"))

dim_store = pl.scan_parquet(os.path.join(PATH_PARQUETS, "dim_store_v2.parquet"))
dim_product = pl.scan_parquet(os.path.join(PATH_PARQUETS, "dim_product_v2.parquet"))
dim_campaign = pl.scan_parquet(os.path.join(PATH_PARQUETS, "dim_campaign_v2.parquet"))
dim_member = pl.scan_parquet(os.path.join(PATH_PARQUETS, "dim_member_v2.parquet"))

In [23]:
tables = {
    "fact_sales": fact_sales,
    "fact_campaign_sales": fact_campaign_sales,
    "fact_points": fact_points,
    "dim_store": dim_store,
    "dim_product": dim_product,
    "dim_campaign": dim_campaign,
    "dim_member": dim_member,
}

for name, df in tables.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.collect_schema())


fact_sales
----------
Schema([('matas_id', String), ('transaction_id', String), ('store_id', String), ('channel', String), ('order_id', String), ('order_delivery_method', String), ('sales_date', Date), ('product_id', String), ('product_quantity', Int32), ('revenue', Float64), ('discount', Float64)])

fact_campaign_sales
-------------------
Schema([('campaign_id', String), ('matas_id', String), ('sales_date', Date), ('store_id', String), ('channel', String), ('transaction_id', String), ('product_id', String), ('revenue', Float64), ('product_quantity', Int32)])

fact_points
-----------
Schema([('matas_id', String), ('point_date', Date), ('point_type', String), ('points', Int32)])

dim_store
---------
Schema([('store_id', String), ('store_name', String), ('store_zip_code', String), ('store_region', String), ('is_webshop', Boolean)])

dim_product
-----------
Schema([('product_id', String), ('product_name', String), ('brand_name', String), ('category', String), ('sub_category', String), ('

### Pandas testing

Testing a full load\.

In [5]:
'''
import psutil
import threading
process = psutil.Process(os.getpid())

peak_memory = 0
monitoring = True

def monitor_memory():
    global peak_memory
    while monitoring:
        mem = process.memory_info().rss / (1024 ** 2)  # MB
        if mem > peak_memory:
            peak_memory = mem
        time.sleep(0.1)

peak_memory = 0
monitoring = True

monitor_thread = threading.Thread(target=monitor_memory)
monitor_thread.start()

start = time.perf_counter()

df = pd.read_parquet("/work/data/parquets/fact_sales_v2.parquet")

end = time.perf_counter()

monitoring = False
monitor_thread.join()

print(f"Load time: {end - start:.2f} seconds")
print(f"Peak RAM during load: {peak_memory:.2f} MB")
print(df.info(memory_usage='deep'))
'''

'\nimport psutil\nimport threading\nprocess = psutil.Process(os.getpid())\n\npeak_memory = 0\nmonitoring = True\n\ndef monitor_memory():\n    global peak_memory\n    while monitoring:\n        mem = process.memory_info().rss / (1024 ** 2)  # MB\n        if mem > peak_memory:\n            peak_memory = mem\n        time.sleep(0.1)\n\npeak_memory = 0\nmonitoring = True\n\nmonitor_thread = threading.Thread(target=monitor_memory)\nmonitor_thread.start()\n\nstart = time.perf_counter()\n\ndf = pd.read_parquet("/work/data/parquets/fact_sales_v2.parquet")\n\nend = time.perf_counter()\n\nmonitoring = False\nmonitor_thread.join()\n\nprint(f"Load time: {end - start:.2f} seconds")\nprint(f"Peak RAM during load: {peak_memory:.2f} MB")\nprint(df.info(memory_usage=\'deep\'))\n'

\>\>\> Result: Depends on when I run this\. Sometimes it runs and hits 4700MB of memory, but mostly is just crashes\. 

### Polars testing

Polars is built for modern hardware and large datasets. Pandas was not.
- Polars typically uses 2–4x less RAM
- Often the difference between crashing and working

In [6]:
import polars as pl
import time

start = time.perf_counter()

df = pl.scan_parquet("/work/data/parquets/fact_sales_v2.parquet")

end = time.perf_counter()

print(f"Lazy scan setup time: {end - start:.2f} seconds")
print(df)

Lazy scan setup time: 0.00 seconds
naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Parquet SCAN [/work/data/parquets/fact_sales_v2.parquet]
PROJECT */11 COLUMNS
ESTIMATED ROWS: 12954168


Testing aggregation \(should be safe\)

In [7]:
start = time.perf_counter()

result = (
    df
    .group_by("store_id")
    .agg(pl.col("revenue").sum())
    .collect()
)

end = time.perf_counter()

print(result)
print(f"Aggregation time: {end - start:.2f} seconds")

shape: (266, 2)
┌──────────┬────────────┐
│ store_id ┆ revenue    │
│ ---      ┆ ---        │
│ str      ┆ f64        │
╞══════════╪════════════╡
│ 15288    ┆ 1.1948e7   │
│ 14427    ┆ 1.0666e7   │
│ 12312    ┆ 3.408094e6 │
│ 18627    ┆ 1.0684e6   │
│ 19275    ┆ 2.9588e6   │
│ …        ┆ …          │
│ 10320    ┆ 5.6599e6   │
│ 12788    ┆ 6.7898e6   │
│ 14200    ┆ 6.9726e6   │
│ 18940    ┆ 3.0650e6   │
│ 19429    ┆ 1.7667e6   │
└──────────┴────────────┘
Aggregation time: 16.39 seconds


Testing full load

In [8]:
start = time.perf_counter()

df_full = pl.read_parquet("/work/data/parquets/fact_sales_v2.parquet")

end = time.perf_counter()

print(f"Full load time: {end - start:.2f} seconds")
print(df_full.estimated_size("mb"), "MB estimated memory usage")

Full load time: 7.59 seconds
1715.7143087387085 MB estimated memory usage


That is an excellent result.
- 12.5 million rows
- Full load in ~3 seconds
- ~1.83 GB estimated memory usage
In a 5GB Deepnote environment, that is completely safe.


Testing full load of web sessions \- the by far the largest file\.

In [9]:
'''
start = time.perf_counter()

df_full = pl.read_parquet("/work/data/parquets/fact_web_sessions_v2.parquet")

end = time.perf_counter()

print(f"Full load time: {end - start:.2f} seconds")
print(df_full.estimated_size("mb"), "MB estimated memory usage")
'''

'\nstart = time.perf_counter()\n\ndf_full = pl.read_parquet("/work/data/parquets/fact_web_sessions_v2.parquet")\n\nend = time.perf_counter()\n\nprint(f"Full load time: {end - start:.2f} seconds")\nprint(df_full.estimated_size("mb"), "MB estimated memory usage")\n'

\>\>\> We can't load the whole web sessions parquet, it overloads the RAM\. 

### Results

We should try using polars for data analysis \(EXCEPT FOR WEB SESSIONS\)\. If that fails, we can't possibly load whole datasets into memory, even from parquets\. 

What we always use to do is "Load everything → then analyze"

We would then need to move towards what data teams usually do: “Query only what I need → then load the result”

Example:

In [10]:
start = time.perf_counter()

result = con.execute("""
    SELECT store_id, SUM(revenue) AS total_revenue
    FROM '/work/data/parquets/fact_sales_v2.parquet'
    GROUP BY store_id
""").fetch_df()

end = time.perf_counter()

print(result.head())
print(f"Aggregation time: {end - start:.2f} seconds")
print(f"Result rows: {len(result)}")

  store_id  total_revenue
0    14616   1.045381e+07
1    16950   5.031054e+06
2    11711   5.075984e+06
3    12874   8.645078e+06
4    10049   3.528035e+06
Aggregation time: 0.92 seconds
Result rows: 266


In [11]:
start = time.perf_counter()

result = con.execute("""
    SELECT store_id, product_id, SUM(revenue) AS total_revenue
    FROM '/work/data/parquets/fact_sales_v2.parquet'
    GROUP BY store_id, product_id
""").fetch_df()

end = time.perf_counter()

print(f"Aggregation time: {end - start:.2f} seconds")
print(f"Result rows: {len(result)}")

Aggregation time: 2.02 seconds
Result rows: 2030911


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1615b33a-d424-41db-9b01-6976e7db6ad0' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>